# The Transformer, and the experiment that reveals positional embeddings

Encoder and decoder blocks built from scratch, trained — and failing — then fixed with two changed lines. The failure is the point.

**Runs on:** GPU recommended — about 45 minutes on CPU &nbsp;·&nbsp; **Slides:** [Chapter 15 — Language Models and the Transformer](../../../course-web-slides/ch15/index.html) &nbsp;·&nbsp; **Section:** 03 — The Transformer architecture

---

## The encoder block

In [ ]:
import keras
from keras import layers, ops

class TransformerEncoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, source, source_mask):
        residual = x = source
        mask = source_mask[:, None, :]
        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)
        x = x + residual
        x = self.self_attention_layernorm(x)

        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

Two stages, each *transform, add the residual, normalise* — the chapter-9 pattern exactly. `key_dim = hidden_dim // num_heads` keeps total width constant as heads are added: **splitting, not growing.**

## LayerNormalization, not BatchNormalization

In [ ]:
import numpy as np

def layer_normalization(batch_of_sequences):
    mean = np.mean(batch_of_sequences, keepdims=True, axis=-1)
    variance = np.var(batch_of_sequences, keepdims=True, axis=-1)
    return (batch_of_sequences - mean) / np.sqrt(variance + 1e-6)

def batch_normalization(batch_of_images):
    mean = np.mean(batch_of_images, keepdims=True, axis=(0, 1, 2))
    variance = np.var(batch_of_images, keepdims=True, axis=(0, 1, 2))
    return (batch_of_images - mean) / np.sqrt(variance + 1e-6)

seqs = np.random.normal(size=(4, 10, 8))
print("layer norm pools over axis -1 only:")
print("  each sequence normalized independently ->",
      layer_normalization(seqs)[0].std().round(3))
print()
print("batch norm pools over axis 0, creating interactions BETWEEN samples.")
print("Sequences in a batch have different lengths and padding amounts,")
print("so batch statistics are contaminated by how you grouped examples.")

## The decoder block

In [ ]:
class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.cross_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.cross_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, target, source, source_mask):
        residual = x = target
        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = self.self_attention_layernorm(x + residual)

        residual = x
        mask = source_mask[:, None, :]
        x = self.cross_attention(query=x, key=source, value=source,
                                 attention_mask=mask)
        x = self.cross_attention_layernorm(x + residual)

        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = self.feed_forward_layernorm(x + residual)
        return x

> ⚠️ **Two different masks, solving two different problems.** `use_causal_mask=True` stops the decoder seeing its own future — notebook 01's bidirectional failure, prevented. The padding `attention_mask` on cross-attention stops it attending to empty source positions.

## Seeing the causal mask

In [ ]:
import matplotlib.pyplot as plt

n = 8
causal = np.tril(np.ones((n, n)))
plt.figure(figsize=(4.5, 4))
plt.imshow(causal, cmap="Greens")
plt.xlabel("source position (key)"); plt.ylabel("target position (query)")
plt.title("Row i may attend to positions 0..i")
plt.colorbar(); plt.show()
print(causal.astype(int))

## The model, first attempt

In [ ]:
hidden_dim, intermediate_dim, num_heads = 256, 2048, 8
vocab_size, sequence_length = 15000, 20

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x, source_mask=source != 0)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x, source=encoder_output, source_mask=source != 0)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)

print(f"{transformer.count_params():,} parameters "
      f"(the GRU model had 34,869,912)")

In [ ]:
transformer.compile(optimizer="adam",
                    loss="sparse_categorical_crossentropy",
                    weighted_metrics=["accuracy"])
h1 = transformer.fit(train_ds, epochs=15, validation_data=val_ds, verbose=2)
print(f"\nbest validation accuracy: {max(h1.history['val_accuracy']):.4f}")
print("the GRU model reached 0.65")

Expected output:

```
best validation accuracy: 0.58xx
```

## Stop here. Why is it worse?

Seven percentage points **worse** than the RNN, with half the parameters. Before reading on, look at the model definition again.

*(Hint: this section is about sequence models. Is the model above a sequence model?)*

## The demonstration

In [ ]:
# Shuffle the words in every source sentence and see what changes.
import tensorflow as tf

for inputs, targets, weights in train_ds.take(1):
    eng = inputs["english"]
    shuffled = tf.random.shuffle(tf.transpose(eng))
    shuffled = tf.transpose(shuffled)

    normal = transformer.predict(
        [eng, inputs["spanish"]], verbose=0)
    scrambled = transformer.predict(
        [shuffled, inputs["spanish"]], verbose=0)

    print("Mean absolute difference in predictions after shuffling")
    print("every word of every source sentence:")
    print(f"  {np.abs(normal - scrambled).mean():.6f}")
    break

Very small, and it would be **exactly zero** if the shuffle were applied consistently within the batch.

The model is dense layers processing tokens independently plus an attention layer that sees tokens **as a set**. Change the order and you get identical pairwise scores. ==Attention is a set-processing mechanism, blind to position.==

## Positional embeddings

In [ ]:
class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs):
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

`ops.cumsum(ops.ones_like(inputs), axis=-1) - 1` produces `[0, 1, 2, …]` — a backend-agnostic `arange`. The layer is a **drop-in replacement** for `Embedding`.

## Two lines changed

In [ ]:
source = keras.Input(shape=(None,), dtype="int32", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x, source_mask=source != 0)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x, source=encoder_output, source_mask=source != 0)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)

transformer.compile(optimizer="adam",
                    loss="sparse_categorical_crossentropy",
                    weighted_metrics=["accuracy"])
import time
t0 = time.time()
h2 = transformer.fit(train_ds, epochs=30, validation_data=val_ds, verbose=2)
print(f"\nbest validation accuracy: {max(h2.history['val_accuracy']):.4f}")
print(f"seconds per epoch: {(time.time()-t0)/30:.1f}")

Expected output:

```
best validation accuracy: 0.67xx
```

## The three numbers, and the one that matters most

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4.4))
plt.plot(h1.history["val_accuracy"], lw=1.6, label="Transformer, no positions")
plt.plot(h2.history["val_accuracy"], lw=1.6, label="Transformer + positions")
plt.axhline(0.65, ls="--", c="k", lw=1.3, label="GRU seq2seq")
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend()
plt.title("Two lines changed")
plt.show()

print("GRU:                    0.65   34.9 M parameters")
print("Transformer, no pos:    0.58   14.4 M parameters")
print("Transformer + pos:      0.67   14.4 M parameters")
print()
print("And each epoch takes about a THIRD the time of the GRU --")
print("no looped state passing, so the whole attention computation")
print("happens in one go on a GPU.")

**That speed-up is the consequential number**, not the accuracy. Parallelisable training is what made scaling to billions of parameters economically possible — chapter 16 is the consequence.

---

## What to take away

- The encoder and decoder blocks are attention plus feedforward, each with add-and-norm.
- `use_causal_mask=True` on decoder self-attention; a padding mask on cross-attention.
- **Attention is order-blind** — shuffle the input and nothing changes.
- Positional embeddings cost two lines; the parallelisability is what changed the field.